# Source registry

Control table `meta.source_registry` in the Bronze lakehouse drives the `Refresh_Orchestration` pipeline.
The pipeline loops over every **enabled** source and runs its Bronze ingestion notebook followed by its Silver
transformation notebook.

**Add a new data source**
1. Create `<Source>_Ingest` (lands raw data into Bronze schema `<source_system>`) and `<Source>_Silver_Transform`
   (writes the source's slice of the Silver tables with `source_system = '<source_system>'`).
2. Set `is_enabled = true` for the source in `meta.source_registry` (the planned rows below already exist), e.g.
   `UPDATE meta.source_registry SET is_enabled = true WHERE source_system = 'garmin_connect'`.
3. Optionally run only some sources by passing `sources_filter` (comma separated) from the pipeline.

Rows are inserted once; edits made directly in the table are preserved.

In [ ]:
sources_filter = ""

In [ ]:
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
import sempy.fabric as fabric

REGISTRY = "meta.source_registry"
REGISTRY_DDL = (
    "source_system string, source_name string, bronze_schema string, ingest_notebook string, silver_notebook string, "
    "is_enabled boolean, load_order int, description string, updated_at timestamp"
)
now = datetime.now(timezone.utc).replace(microsecond=0)
DEFAULT_SOURCES = [
    ("strava", "Strava", "strava", "Strava_Ingest", "Strava_Silver_Transform", True, 10,
     "Strava API v3: activity summaries, detailed activities, streams, athlete profile and zones", now),
    ("garmin_connect", "Garmin Connect", "garmin_connect", "Garmin_Ingest", "Garmin_Silver_Transform", False, 20,
     "Planned: Garmin Health API - activities, sleep, daily summaries, HRV, body battery, stress", now),
    ("apple_health", "Apple Health", "apple_health", "Apple_Health_Ingest", "Apple_Health_Silver_Transform", False, 30,
     "Planned: HealthKit data uploaded by the iOS app - workouts, sleep, biometrics, nutrition", now),
]

spark.sql("CREATE SCHEMA IF NOT EXISTS meta")
if not spark.catalog.tableExists(REGISTRY):
    spark.createDataFrame([], REGISTRY_DDL).write.format("delta").saveAsTable(REGISTRY)

spark.createDataFrame(DEFAULT_SOURCES, REGISTRY_DDL).createOrReplaceTempView("default_sources")
spark.sql(f"""
    MERGE INTO {REGISTRY} AS t
    USING default_sources AS s
    ON t.source_system = s.source_system
    WHEN NOT MATCHED THEN INSERT *
""")

enabled = spark.table(REGISTRY).where("is_enabled")
if sources_filter.strip():
    enabled = enabled.where(F.col("source_system").isin([s.strip() for s in sources_filter.split(",") if s.strip()]))
enabled_rows = enabled.orderBy("load_order", "source_system").collect()

notebooks = fabric.list_items(type="Notebook")
notebook_ids = dict(zip(notebooks["Display Name"], notebooks["Id"]))
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

sources = []
for row in enabled_rows:
    missing = [n for n in (row["ingest_notebook"], row["silver_notebook"]) if n not in notebook_ids]
    if missing:
        raise RuntimeError(f"Source '{row['source_system']}' is enabled but notebooks are missing: {missing}")
    sources.append({
        "source_system": row["source_system"],
        "source_name": row["source_name"],
        "bronze_schema": row["bronze_schema"],
        "workspace_id": workspace_id,
        "ingest_notebook_id": notebook_ids[row["ingest_notebook"]],
        "silver_notebook_id": notebook_ids[row["silver_notebook"]],
    })

if not sources:
    raise RuntimeError("No enabled sources in meta.source_registry")

print(json.dumps(sources, indent=2))
notebookutils.notebook.exit(json.dumps(sources))